# 6 · Risk-Adjusted Metrics vs Nifty 50

Recomputes Beta, Jensen's Alpha and Treynor Ratio benchmarked against the  
**actual Nifty 50 index** (monthly returns, aligned to the portfolio date range).

**Key fixes vs original code:**
- RF rate is `RF_MONTHLY` (6.5% ÷ 12), not `0.05 / 252` applied to monthly data  
- All three portfolios (Equal, MaxSharpe, MinVar) are evaluated  
- Date alignment uses an inner join to avoid NaN contamination  
- The Pandas `FutureWarning` on `concat` sort is suppressed via `sort=True`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import (
    DATA_DIR, ANALYSIS_DIR, RF_MONTHLY, RF_ANNUAL, PERIODS_PER_YEAR,
    beta, jensen_alpha, treynor_ratio,
    sortino_ratio, max_drawdown, calmar_ratio,
    annualise_return, annualise_vol
)

data_path     = Path(DATA_DIR)
analysis_path = Path(ANALYSIS_DIR)

# ── Portfolio returns ─────────────────────────────────────────────────────────
returns = pd.read_csv(
    analysis_path / "india_equity_returns.csv",
    index_col="Date", parse_dates=True
)
weights = pd.read_csv(analysis_path / "portfolio_weights.csv", index_col="Asset")

w_equal  = weights["Equal_Weight"].values
w_opt    = weights["MaxSharpe_Weight"].values
w_minvar = weights["MinVar_Weight"].values

port_equal  = returns @ w_equal
port_opt    = returns @ w_opt
port_minvar = returns @ w_minvar

# ── Load Nifty 50 ─────────────────────────────────────────────────────────────
nifty_files = list(data_path.glob("Nifty*"))
assert len(nifty_files) == 1, f"Expected 1 Nifty file, found {nifty_files}"

nifty = pd.read_csv(nifty_files[0])
nifty.columns = [c.strip() for c in nifty.columns]
nifty["Date"]  = pd.to_datetime(nifty["Date"])
nifty = nifty.sort_values("Date").set_index("Date")
nifty["Price"] = (
    nifty["Price"].astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

nifty_returns = nifty["Price"].pct_change().dropna()
nifty_returns.name = "NIFTY"

# ── Inner-join on shared dates (no NaN, no sort warning) ─────────────────────
combined = pd.concat(
    [port_equal.rename("Equal"), port_opt.rename("MaxSharpe"),
     port_minvar.rename("MinVar"), nifty_returns],
    axis=1, join="inner", sort=True
).dropna()

print(f"Aligned date range: {combined.index.min().date()} → {combined.index.max().date()}")
print(f"Observations: {len(combined)}")

market = combined["NIFTY"]

# ── Compute metrics ───────────────────────────────────────────────────────────
def nifty_metrics(label, port_col):
    port = combined[port_col]
    b    = beta(port, market)
    sr   = (port.mean() - RF_MONTHLY) / port.std()
    return {
        "Portfolio"          : label,
        "Monthly_Return"     : round(port.mean(), 6),
        "Annual_Return"      : round(annualise_return(port.mean()), 6),
        "Monthly_Vol"        : round(port.std(), 6),
        "Annual_Vol"         : round(annualise_vol(port.std()), 6),
        "Sharpe_Monthly"     : round(sr, 6),
        "Sharpe_Annual"      : round(sr * np.sqrt(PERIODS_PER_YEAR), 6),
        "Sortino_Monthly"    : round(sortino_ratio(port, RF_MONTHLY), 6),
        "Max_Drawdown"       : round(max_drawdown(port), 6),
        "Calmar"             : round(calmar_ratio(port), 6),
        "Beta_Nifty"         : round(b, 6),
        "Jensen_Alpha_Mo"    : round(jensen_alpha(port, market, b, RF_MONTHLY), 6),
        "Treynor_Monthly"    : round(treynor_ratio(port, b, RF_MONTHLY), 6),
        "Nifty_Monthly_Ret"  : round(market.mean(), 6),
        "Nifty_Annual_Ret"   : round(annualise_return(market.mean()), 6),
    }

metrics = pd.DataFrame([
    nifty_metrics("Equal Weight", "Equal"),
    nifty_metrics("Max Sharpe",   "MaxSharpe"),
    nifty_metrics("Min Variance", "MinVar"),
])

metrics.to_csv(analysis_path / "risk_adjusted_metrics.csv", index=False)

print(f"\nRF (monthly) : {RF_MONTHLY*100:.4f}%  [{RF_ANNUAL*100:.1f}% p.a. ÷ {PERIODS_PER_YEAR}]")
print("\nRisk-Adjusted Metrics (Nifty 50 benchmark):")
print(metrics.to_string(index=False))
print("\nSaved: analysis/risk_adjusted_metrics.csv")

Aligned date range: 2020-01-01 → 2026-01-01
Observations: 7

RF (monthly) : 0.5417%  [6.5% p.a. ÷ 12]

Risk-Adjusted Metrics (Nifty 50 benchmark):
   Portfolio  Monthly_Return  Annual_Return  Monthly_Vol  Annual_Vol  Sharpe_Monthly  Sharpe_Annual  Sortino_Monthly  Max_Drawdown    Calmar  Beta_Nifty  Jensen_Alpha_Mo  Treynor_Monthly  Nifty_Monthly_Ret  Nifty_Annual_Ret
Equal Weight       -0.006938      -0.080150     0.014739    0.051056       -0.838247      -2.903771        -0.630494     -0.040998 -1.954981    0.973931         0.007401        -0.012685          -0.014867         -0.164519
  Max Sharpe        0.018223       0.241986     0.044547    0.154317        0.287485       0.995877         0.283698     -0.047699  5.073200    2.548082         0.064492         0.005026          -0.014867         -0.164519
Min Variance       -0.002670      -0.031576     0.019948    0.069101       -0.405402      -1.404355        -0.384770     -0.047842 -0.660002    0.783503         0.007806        -0.0